# CITE-seq Surface Protein Analysis with MANTRA

This tutorial applies MANTRA to CITE-seq (Cellular Indexing of Transcriptomes
and Epitopes by Sequencing) surface protein abundance data. The dataset
contains 120,502 cells from 12 donors across 2 batches, measuring 136 surface
proteins.

Unlike the RNA tutorial which started from annotated cell types, here we will:

1. Cluster cells to discover cell types de novo
2. Build a pseudo-bulk tensor (donors x cell types x proteins)
3. Fit MANTRA on this compact but informative feature space
4. Interpret protein factors and donor heterogeneity

## 1. Load and Explore the Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import mantra

sc.settings.verbosity = 1

In [ ]:
adata = sc.read_h5ad("cite_preprocessed.h5ad")
print(adata)
print(f"\nDonors: {adata.obs['donor'].nunique()} ({adata.obs['donor'].unique().tolist()})")
print(f"Batches: {adata.obs['batch'].nunique()}")
print(f"Proteins: {adata.n_vars}")
print(f"\nProtein names (first 20): {adata.var_names[:20].tolist()}")

## 2. Discover Cell Types via Clustering

The dataset doesn't include cell type annotations, so we cluster cells using
the precomputed PCA and neighbor graph.

In [ ]:
# The data has precomputed PCA and neighbors -- recompute neighbors to ensure
# consistency, using the existing PCA
sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=30)
sc.tl.leiden(adata, resolution=0.8, key_added="cell_type")

print(f"Discovered {adata.obs['cell_type'].nunique()} cell type clusters")
adata.obs["cell_type"].value_counts()

In [ ]:
sc.tl.umap(adata)
sc.pl.umap(adata, color=["cell_type", "donor"], ncols=2, show=True)

## 3. Build Pseudo-bulk Tensor

Aggregate the 120K cells into a 3D tensor: **donors x cell_types x proteins**.
This gives us a compact tensor that MANTRA can decompose efficiently.

In [ ]:
tensor, metadata = mantra.pp.pseudobulk(
    adata,
    sample_key="donor",
    slice_key="cell_type",
    agg_func="mean",
    min_cells=20,
)

print(f"Tensor shape: {tensor.shape}  (donors x cell_types x proteins)")
print(f"Donors: {metadata['sample_names']}")
print(f"Cell types (clusters): {metadata['slice_names']}")

# Check missingness
import torch

n_missing = torch.isnan(tensor[:, :, 0]).sum().item()
n_total = tensor.shape[0] * tensor.shape[1]
print(f"\nMissing groups: {n_missing}/{n_total} ({100*n_missing/n_total:.1f}%)")

## 4. Normalize and Fit MANTRA

In [ ]:
tensor_norm = mantra.pp.normalize(tensor, center=True, scale=True)

model = mantra.MANTRA(
    observations=tensor_norm,
    R=8,
    use_gpu=False,
)

model.sample_names = metadata["sample_names"]
model.slice_names = metadata["slice_names"]
model.feature_names = [metadata["feature_names"]]

print(model)

In [ ]:
from mantra.inference.callbacks import EarlyStoppingCallback

history, stopped_early = model.fit(
    n_epochs=3000,
    learning_rate=0.005,
    seed=42,
    callbacks=[EarlyStoppingCallback(patience=100)],
)
print(f"\nFinal ELBO: {history[-1]:.2f}")
print(f"Epochs run: {len(history)}")

In [ ]:
fig = mantra.pl.plot_elbo(history)
plt.show()

## 5. Variance Explained

In [ ]:
r2 = mantra.tl.variance_explained(model)
print(f"Total R-squared: {r2['total']:.4f}\n")
print(r2["per_factor"])

In [ ]:
fig = mantra.pl.variance_explained(model)
plt.show()

## 6. Interpret Protein Factors

### 6a. Cell Type Loadings (A2)

Which cell type clusters drive each factor?

In [ ]:
top_factors = mantra.tl.filter_factors(model, r2_thresh=0.90)
top_factor_idx = list(range(min(3, len(top_factors))))

fig = mantra.pl.slice_weights(model, factor_idx=top_factor_idx)
plt.show()

### 6b. Protein Loadings per Factor

The heatmap shows cell_type x protein loadings. Top proteins per factor
reveal which surface markers define that factor's biology.

In [ ]:
for fi in top_factor_idx:
    fig = mantra.pl.factor_weights(model, factor_idx=fi, top=20)
    plt.show()

### 6c. Top Proteins per Factor

Extract the highest-loading proteins for each factor.

In [ ]:
A3 = model.get_feature_embeddings(as_df=True)

for fi in top_factor_idx:
    factor_name = model.factor_names[fi]
    loadings = A3[factor_name].abs().sort_values(ascending=False)
    print(f"\n{factor_name} - Top 10 proteins:")
    for gene, val in loadings.head(10).items():
        sign = "+" if A3.loc[gene, factor_name] > 0 else "-"
        print(f"  {sign} {gene}: {val:.4f}")

## 7. Donor Embeddings

### 7a. Factor Scatter Plot

In [ ]:
# Add batch info as metadata
batch_map = adata.obs.groupby("donor")["batch"].first()
batch_labels = [str(batch_map.get(d, "?")) for d in metadata["sample_names"]]
mantra.tl.add_metadata(model, "batch", batch_labels)

fig = mantra.pl.scatter(model, x=0, y=1, color="batch")
plt.show()

### 7b. Donor UMAP

In [ ]:
n_donors = len(metadata["sample_names"])
mantra.tl.neighbors(model, n_neighbors=min(5, n_donors - 1))
mantra.tl.umap(model)

fig = mantra.pl.embedding(model, color="batch", method="umap")
plt.show()

### 7c. Hierarchical Clustering of Donors

In [ ]:
g = mantra.pl.clustermap(model)
plt.show()

### 7d. Factor Values by Batch

In [ ]:
fig = mantra.pl.boxplot(model, factor_idx=top_factor_idx, groupby="batch")
plt.show()

## 8. Reconstruction Quality

In [ ]:
Y_hat = model.get_reconstructed()
rmse = mantra.tl.rmse_loss(Y_hat, tensor_norm)
print(f"Reconstruction RMSE: {rmse:.4f}")

## Summary

In this tutorial we applied MANTRA to CITE-seq surface protein data:

- **Discovered cell types** via Leiden clustering on protein profiles
- **Built a pseudo-bulk tensor** (12 donors x N clusters x 136 proteins)
- **Identified protein factors** that capture donor and cell type variation
- **Interpreted factors** through protein loadings and cell type weights

The compact protein feature space (136 features vs 20K genes) makes this an
efficient use case for MANTRA, with factors directly interpretable through
known surface markers (CD3, CD19, CD8, etc.).

Next: [04_advanced_features.ipynb](04_advanced_features.ipynb) covers callbacks,
save/load, factor filtering, and other advanced features.